<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/01-model-apis/00-prompting-basics.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompt 基础

**目标：** 在动用任何 API 特性*之前*，先从模型身上多榨出成果的四种技术：清晰的指令、示例、输出格式规范，以及分步推理。每一种都用真实数字展示效果，而不是凭感觉。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

## Prompt 是*为了*什么

在讲技术之前，先弄清楚*为什么*值得在 prompt 上下功夫。好的 prompt 不是为了让模型觉得你很聪明。它是三个方面的最便宜的杠杆——否则你要用代码、延迟或真金白银去为它们买单：

| 目标 | 更好的 prompt 能换来什么 | 课程在哪里深入展开 |
|---|---|---|
| **质量与可靠性** | 把正确答案*稳定地*拿到，而不是偶尔拿到一次：错误输出更少，多次调用之间的漂移更小 | 在 **eval**（第 02、04 节）里被正确度量 |
| **格式 / 结构** | 输出你的*程序*能消费的内容，形状固定，而不是一段要拿正则去抠的散文 | prompting 能让你接近目标；**01-structured-output** 把它变成保证 |
| **成本与延迟** | 用**更少的 token** 拿到同样的结果。一句把任务讲清楚的紧凑 prompt，胜过一段过度解释、还把最终要丢弃的输出也一并生成的啰嗦 prompt | 在 **04-context-and-caching** 里做预算 |

最后一条常被低估：**精确的 prompt 往往就是更便宜的 prompt。** 用「一个词作答」代替让模型写一整段，或者只索取你真正会用到的字段，都会在*每一次*调用上削减输出 token，而调用次数以百万计时，省下来的就是成倍的钱。prompt 的质量和成本管控，是同一个杠杆从两侧发力。

本 notebook 剩下的部分讲的是*怎么做*：四种服务于这些目标的技术，每一种都用真实数字展示。

## 准备工作

每个 notebook 都是自包含的，所以接下来两个单元格会从零把它搭起来：

1. **安装依赖。** `aien` 包（本仓库）带着共享的 setup 辅助函数，并拉入 `groq` 客户端——这是本 notebook 唯一需要的依赖。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领免费 key（无需信用卡）；在 Colab 里通过**钥匙图标** → **Add new secret** 添加，名称严格写成 `GROQ_API_KEY`，打开 notebook access。本地则在环境变量里设置 `GROQ_API_KEY`。

（完整讲解见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup

# 读取 GROQ_API_KEY（Colab Secrets 或本地环境变量），返回一个就绪的 Groq 客户端。
client, MODEL = setup()

## prompt 就是你的首要 API 界面

在 RAG 之前，在 agent 之前，在 fine-tuning 之前，对模型行为而言最便宜、最快的杠杆就是发给它的那些文字。这就是 *prompting*，值得认真对待，因为这门课里的其他每一项技术内部都在做这件事：RAG 把检索到的文本塞进 prompt，agent 的整套人格就是一段 prompt，judge 也是一段 prompt。

先澄清两件事，因为「prompt engineering」这个词会招来一堆胡话：

> **🚩 常见错误 ——** 照猫画虎式的 prompting：塞一些咒语式短语（「你是一位世界级专家」「我会给你 200 美元小费」「深呼吸」）。其中一些在老模型上有效，在当前模型上基本失效。不要去收集咒语。学那几个有*机制*的技术，只留下真正能改善你指标的那些。

还有一条让这一切落地的纪律，沿袭自**第 02 节（eval）**：prompt 的改动是*被度量*的改动。「我试的那个例子上它看起来更好了」，正是你把一个在其余九十个例子上更差的 prompt 发上线的路径。下面每种技术都给了前后对比，你可以把它变成一次 eval。本 notebook 用肉眼对比来教你机制；实践中你要拿黄金集给它打分。

## 技术 1：把你要的东西说清楚

最有力的单一杠杆，也是最常被浪费的。模型有点较真：含糊的指令得到含糊、不一致的输出。把**任务、格式和约束**明确写出来。看看同一个输入上，一个偷懒的 prompt 与一个写清楚的 prompt 的差别。

In [ ]:
review = ("Bought the standing desk. Assembly took two people and the instructions "
          "were useless, but it's rock solid and the motor is whisper quiet. "
          "Shipping was a week late though.")

def ask(prompt):
    r = client.chat.completions.create(
        model=MODEL, max_tokens=200,
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

print("--- vague:")
print(ask(f"What do you think of this review?\n\n{review}"))

print("\n--- specified (task + format + constraint):")
print(ask(
    "Extract, from the product review below, the aspects mentioned and the sentiment "
    "for each. Output one line per aspect as `aspect: sentiment` where sentiment is "
    "positive, negative, or mixed. Only aspects the review actually mentions.\n\n"
    f"{review}"))

含糊的 prompt 给你一段得靠手工解析的散文；写清楚的 prompt 给你结构化、稳定的行。**大多数「模型做不了这个」的问题，其实是「我没把它想要什么讲清楚」。** 修改指令是零成本的，应该永远是你的第一步，排在 few-shot 之前、工具之前、一切之前。

几乎每次都应该明确写出的三件事：
- **任务：** 要做什么，用一个动词（「classify」「extract」「rewrite」）。
- **格式：** 输出的确切形状（每行一条、JSON、单个词）。
- **约束：** 边界（「只写提到的内容」「≤ 20 个词」「不要开场白」）。

注意，写清楚的 prompt 同时也是*更便宜*的那个：它要求紧凑的短行而不是一整段，于是每次调用的输出 token 更少。「每个方面一行」或「一个词作答」这类约束，既是质量控制**也是**成本控制，是同一个杠杆的两面（token 的算术在第 04 节）。

## 技术 2：few-shot，用展示代替只说

当某个任务很难*描述*、却容易*演示*时（一个特定的标签集、一种输出风格、一条边界情况的约定），给模型几个 input → output 的例子。这就是 **few-shot prompting**（zero-shot = 不给例子；few-shot = 给几个）。例子能钉住散文难以言明的行为。

下面是一个小众分类任务（把支持工单路由到内部团队），标签含义光看名字并不明显。Zero-shot 靠猜；few-shot 把它锁定。

In [ ]:
SYSTEM = "Route each ticket to exactly one team. Reply with only the team name."
teams = "Teams: BILLING, PLATFORM (outages/API), DEVICE (hardware), TRUST (fraud/abuse)."

def route(ticket, shots=""):
    r = client.chat.completions.create(
        model=MODEL, max_tokens=10,
        messages=[{"role": "system", "content": f"{SYSTEM} {teams}"},
                  {"role": "user", "content": f"{shots}Ticket: {ticket}\nTeam:"}])
    return r.choices[0].message.content.strip()

# 一张真正含糊的工单：「取消之后仍被扣费」可能读作 BILLING，
# 但这里约定的是，疑似欺诈的扣费归 TRUST。
ticket = "I cancelled last month but was charged again, and I don't recognize the device that logged in."

print("zero-shot:", route(ticket))

FEWSHOT = (
    "Ticket: My invoice is higher than my plan.\nTeam: BILLING\n"
    "Ticket: Charges I don't recognize plus a login from another country.\nTeam: TRUST\n"
    "Ticket: The API returns 503 for every request.\nTeam: PLATFORM\n"
)
print("few-shot :", route(ticket, shots=FEWSHOT))

只要给一个欺诈路由约定的例子，模型就能把它套用到新工单上。用好 few-shot 的几点说明：

- **例子教的是你难以言明的模式：** 语气、边界情况处理、标签分类体系。如果你*能*把规则讲清楚，那就优先用清晰的指令（技术 1）；它更便宜。
- **覆盖边界，而不只是容易的例子。** 真正有价值的例子是那个含糊的。
- **few-shot 在每次调用都要花 token**（例子随每个 prompt 一起发送）。这就是它相比把行为 fine-tuning 进去的取舍，这个决策你会在第 06 节做出。
- 结构化输出（下一个 notebook）常常*取代*格式示例：schema 把形状钉死，你的 few-shot 就可以专注于判断。

## 技术 3：指定输出格式（通往一切结构化内容的桥梁）

如果下游有程序要消费这个输出，你就必须约束它的形状。散文是给人看的；你的代码要的是固定格式。把格式要求明确写出来，能让你走完大部分路，也揭示了下一个 notebook 存在的*原因*。

In [ ]:
bug = ("App crashes on launch after the 4.2 update, but only on Android 13. "
       "Started yesterday. My coworker on iOS is fine.")

print(ask(
    "Summarize this bug report as JSON with keys: title (short string), "
    "platform (string), severity (one of: low, medium, high). "
    "Output only the JSON, no other text.\n\n" + bug))

这通常管用，而*通常*正是问题所在。多跑几次：有时你会得到 markdown 的 ```json 围栏、一句「Here's the JSON:」开场白，或者一个被改写过的 key。当由程序解析结果时，用 prompting 求格式是**必要但不充分**的。

> **⭐ 关键要点 ——** 清晰的格式指令能让你走完 90% 的路；最后 10%（保证可解析、符合 schema 的输出）需要 API 层面的工具——JSON mode 和 tool/function calling，这正是 **[01-structured-output](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/01-model-apis/01-structured-output.zh.ipynb)** 接下来要构建的。prompting 是地板，结构化输出才是保证。

## 技术 4：先要推理，再要答案

对任何需要几步才能完成的事情（算术、逻辑、多约束决策），让模型**先想清楚再作答**（「chain of thought」）能实实在在地提高准确率。它给了模型用来计算的 token，而不是逼它一次性猜出来。代价是：推理过程现在*出现在输出里*了，所以你要么留着它，要么把最终答案抽出来。

下面这道应用题，如果模型立刻作答就可能出错；对比有分步与无分步两种情形：

In [ ]:
q = ("A team of 3 reviews 90 tickets. Two reviewers handle 25 each; the third takes "
     "the rest. If the third reviewer works twice as fast and clears 8 tickets an hour, "
     "how many hours does the third reviewer need? Give the final number.")

print("--- answer-first (no reasoning room):")
print(ask(q + " Reply with just the number."))

print("\n--- reasoning first, then answer:")
print(ask(q + " Think step by step, then end with 'ANSWER: <number>'."))

分步版本更可靠，而且它把答案放在一个可解析的标记（`ANSWER:`）之后，这样你的代码仍能抽出一个字段。两点提醒，让你别把它当条件反射：

- **它要花 token 和延迟：** 推理是你额外生成出来的。别把它加到不需要的任务上（简单分类、抽取）。
- **现代的「reasoning」模型内部就会做这件事。** 在这类模型上，明写「think step by step」常常是多余的，甚至有害。一如既往：这是个 eval 问题，不是信条。去度量它在*你的*任务和模型上有没有帮助。

## 最佳实践与反模式

| ✅ 要做 | ❌ 反模式 |
|---|---|
| 明确写出任务 + 格式 + 约束；先修指令 | 你从没把要求讲清楚，却怪「模型做不了这个」 |
| 当行为更易展示而非描述时用 few-shot；覆盖边界情况 | 一条能一句话说清的规则，却堆一堆例子 |
| 指定输出格式，程序要解析时用结构化输出（nb 01） | 让下游解析器去信任散文式的「输出 JSON」 |
| 只在值得花 token 的地方加分步推理；把最终字段抽出来 | 每次调用都硬塞 CoT，或塞给本来就会做的 reasoning 模型 |
| 只索取你会用到的 token（紧凑输出、必要字段），每次调用更便宜 | 啰嗦的 prompt + 开放式输出，然后再截断或忽略 |
| 用黄金集评判每一次 prompt 改动（第 02 节） | 盯着一个例子调 prompt；照抄咒语 |

（完整全栈清单见 [docs/best-practices-and-anti-patterns.md](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/docs/best-practices-and-anti-patterns.md)。）

## 练习

1. **收紧指令。** 拿技术 1 里那个含糊的评论 prompt，一次加一条约束（格式、最大长度、「只写提到的方面」）。每个版本各跑 3 次，记下哪条约束消除的差异最多。单条改动里哪一条帮助最大？
2. **度量 zero-shot 与 few-shot。** 写 8 张含糊的工单，并给出已知的正确团队（一个小黄金集，第 02 节的风格）。算出 zero-shot 与用你那 3 个示例时的路由准确率。few-shot 真的赢了吗，赢多少？再试 6 个示例：更多是更有帮助，还是到了平台期？
3. **CoT 何时适得其反。** 找一个*简单*任务（单标签情感），对比有与没有「think step by step」时的准确率和 token 成本。确认推理只增加了成本、没带来准确率提升，这就是*别*把它当条件反射的证据。
4. **prompt 是最便宜的杠杆。** 挑一件模型做错的事，*只*用 prompt 改动（指令、示例、格式、推理）把它做对，不写代码、不用工具。记录下什么起了作用。这就是习惯：在伸手去够后面 notebook 里那些昂贵的杠杆（结构化输出、RAG、fine-tuning）之前，先把免费的杠杆用尽。